# 86 — Danh mục nhiều mã trong backtrader

Notebook [`34`](../03-phan-tich-ky-thuat/34_backtest_chien_luoc.ipynb) backtest
chiến lược đà tăng 15 mã bằng **ma trận**: trọng số nhân lợi suất, cộng theo
hàng, xong. Ở đây ta dựng lại **đúng chiến lược đó** bằng backtrader — hàng trăm
feed trong một `Cerebro`, đặt lệnh thật, broker khớp thật — rồi đặt hai đường
vốn cạnh nhau.

Hai bộ máy sẽ không cho cùng một con số. Câu hỏi của notebook này: **khoảng
cách đến từ đâu, và mỗi nguồn đóng góp bao nhiêu?**

| Phần | Nội dung |
|---|---|
| 1 | Tải dữ liệu theo lô — `max_rows`, `truncated`, đơn vị mất sau `pd.concat`, phiên dở dang |
| 2 | Bản vectorised của `34` làm thước chuẩn |
| 3 | Nhiều feed trong một `Cerebro`: `prenext`, `len(data)`, `nextstart` |
| 4 | Chiến lược đà tăng: `order_target_percent`, bán trước mua sau |
| 5 | Đối chiếu hai đường vốn |
| 6 | Truy từng nguồn lệch → bảng đóng góp |
| 7 | Thời gian chạy theo số mã, và giới hạn vũ trụ |

In [1]:
import sys
import time
from pathlib import Path

GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "finlens_examples").is_dir())
sys.path.insert(0, str(GOC))

import backtrader as bt
import numpy as np
import pandas as pd

import finlens
from finlens_examples import ap_dung_theme, bar_ngang, duong, hom_nay, lui_ngay
from finlens_examples.bt_vn import (
    PhiCoPhieuVN,
    bo_phien_chua_dong,
    feed_tu_finlens,
    ket_qua,
    lam_tron_lo,
    them_analyzer,
)

ap_dung_theme()
client = finlens.client()
HOM_NAY = hom_nay(client)

D:\finlens\finlens-python\finlens-python-example\venv\Lib\site-packages\anyio\_core\_tasks.py:275: UpgradeAvailableWarning: finlens 1.5.0 đã có (đang dùng 1.4.0). Nâng cấp: pip install -U finlens
  retval = await self._coro


Luật chơi chép nguyên từ `34` — không đổi một con số nào, vì mục tiêu là đối
chiếu chứ không phải cải thiện. Chỉ thêm một thứ mà ma trận không cần còn
broker thì cần: **vốn bằng đồng**.

In [2]:
SO_NAM = 5
SO_MA_GIU = 15
CUA_SO_DA = 60  # phiên
NGUONG_GTGD = 5e9  # VND/phiên, tính trên 60 phiên TRƯỚC ngày tái cơ cấu
CHI_PHI_MOT_CHIEU = 0.0018  # 0,18% mỗi chiều — giả định của 34
TAN_SUAT_TAI_CO_CAU = "ME"  # cuối mỗi tháng

VON =1e9  # 1 tỷ VND — backtrader cần vốn thật để làm tròn cổ phiếu và lô

## 1 · Dữ liệu — ba chỗ nối sai âm thầm

Tải như `34`: chia lô theo `max_rows ÷ số phiên`, kiểm `truncated` **ở từng
lô**. Khác `34` một điểm: hàm trả về danh sách các lô *chưa ghép*, để ta đọc
đơn vị từ `attrs` của từng lô trước khi `pd.concat` xoá mất chúng.

In [3]:
def tai_theo_lo(ma_ds: list[str], *, start: str, so_phien_uoc_tinh: int) -> list[pd.DataFrame]:
    # Kích thước lô suy từ max_rows của gói; lô nào vẫn bị cắt thì chia đôi.
    max_rows = client.limits()["max_rows"]
    lo = max(10, int(max_rows / so_phien_uoc_tinh * 0.8))
    phan: list[pd.DataFrame] = []

    def lay(ds: list[str]) -> None:
        d = client.eod.stock.ohlcv(ds, start=start)
        if d.attrs["finlens"]["truncated"]:
            if len(ds) == 1:
                raise RuntimeError(f"{ds[0]} vẫn bị cắt — thu hẹp khoảng thời gian")
            giua = len(ds) // 2
            lay(ds[:giua])
            lay(ds[giua:])
            return
        phan.append(d)

    for i in range(0, len(ma_ds), lo):
        lay(ma_ds[i : i + lo])
    return phan


danh_muc = client.meta.symbols(exchange="HOSE", kind="stock")
t0 = time.perf_counter()
phan = tai_theo_lo(danh_muc["symbol"].tolist(), start=lui_ngay(HOM_NAY, nam=SO_NAM), so_phien_uoc_tinh=SO_NAM * 250)
print(f"{len(phan)} lô trong {time.perf_counter() - t0:.0f} giây · lô nào bị cắt: "
      f"{sum(p.attrs['finlens']['truncated'] for p in phan)}")

don_vi_cac_lo = {p.attrs["finlens"]["units"]["close"] for p in phan}
print(f"Đơn vị giá đọc từ attrs của từng lô: {don_vi_cac_lo}")
assert len(don_vi_cac_lo) == 1, "Các lô khác đơn vị — không được ghép"
DON_VI_GIA = don_vi_cac_lo.pop()

gia_ghep = pd.concat(phan, ignore_index=True)
print(f"attrs sau pd.concat: {gia_ghep.attrs!r}")

7 lô trong 148 giây · lô nào bị cắt: 0
Đơn vị giá đọc từ attrs của từng lô: {'kVND'}
attrs sau pd.concat: {}


⚠️ **`attrs` rỗng sau `pd.concat`.** `feed_tu_finlens` đọc đơn vị từ đó để quy
kVND → VND, và khi không còn gì để đọc thì nó **từ chối đoán** — đoán sai là
lệch 1.000 lần mà không có lỗi nào (notebook
[`81`](81_tu_finlens_toi_cerebro.ipynb)). Thử gọi như với một frame vừa tải:

In [4]:
ma_thu = str(gia_ghep["symbol"].iloc[0])
try:
    feed_tu_finlens(gia_ghep, ma=ma_thu)
except ValueError as loi:
    print(f"ValueError: {loi}")

feed_dung = feed_tu_finlens(gia_ghep, ma=ma_thu, don_vi_gia=DON_VI_GIA)
print(f"Truyền don_vi_gia={DON_VI_GIA!r}: giá đóng cửa đầu tiên của {ma_thu} = "
      f"{feed_dung.p.dataname['close'].iloc[0]:,.0f} VND "
      f"(frame gốc ghi {gia_ghep.loc[gia_ghep['symbol'] == ma_thu, 'close'].iloc[0]:,.2f} {DON_VI_GIA})")

ValueError: Frame không còn df.attrs['finlens'] (thường do pd.concat/merge). Truyền don_vi_gia='kVND' | 'VND' | 'index_point' cho rõ.
Truyền don_vi_gia='kVND': giá đóng cửa đầu tiên của AAA = 14,720 VND (frame gốc ghi 14.72 kVND)


Chỗ nối thứ ba: **phiên đang chạy**. Gọi finlens trong giờ giao dịch trả về cả
thanh của hôm nay với giá "đóng cửa" là giá khớp gần nhất. `bo_phien_chua_dong`
bỏ nó theo đồng hồ server — `34` cũng làm vậy, sau khi chính phép đối chiếu
trong notebook này cho thấy bản trước của `34` giữ thanh đó lại.

In [5]:
gia = bo_phien_chua_dong(gia_ghep, client).sort_values(["symbol", "date"], ignore_index=True)
gio_server = client.whoami()["server_time"]
print(f"Giờ server: {gio_server[:16]} · bỏ {len(gia_ghep) - len(gia):,} dòng của phiên chưa đóng cửa")
print(f"Còn {len(gia):,} dòng · {gia['symbol'].nunique()} mã · "
      f"{gia['date'].min():%d/%m/%Y} → {gia['date'].max():%d/%m/%Y}")

HE_SO_VND = {"kVND": 1_000.0, "VND": 1.0}[DON_VI_GIA]
gia["gtgd"] = gia["close"] * HE_SO_VND * gia["volume"]

M_GIA = gia.pivot(index="date", columns="symbol", values="close")
M_MO = gia.pivot(index="date", columns="symbol", values="open")
M_GTGD = gia.pivot(index="date", columns="symbol", values="gtgd")
print(f"Ma trận: {M_GIA.shape[0]} phiên × {M_GIA.shape[1]} mã")

Giờ server: 2026-10-20T13:30 · bỏ 406 dòng của phiên chưa đóng cửa
Còn 490,395 dòng · 409 mã · 20/10/2021 → 19/10/2026


Ma trận: 1245 phiên × 409 mã


## 2 · Thước chuẩn: hàm `chay_backtest` của `34`, chép nguyên

Ô dưới là hàm `chay_backtest` của notebook `34` — **chép nguyên, không sửa một
dòng**, để thước chuẩn ở đây đúng là bản `34` người đọc thấy. Mặc định của nó
là "bản đúng": khớp ở giá mở cửa phiên `t+1`, trọng số trôi theo giá trong
tháng, lấp giá qua phiên tạm ngừng, 0,18% mỗi chiều.

Bản trước của `34` có bốn lỗi mà chính phép đối chiếu trong notebook này đã lôi
ra: giữ thanh phiên đang chạy, mất lợi suất qua phiên tạm ngừng, trọng số cố
định 1/15 mỗi ngày (tức tái cân bằng miễn phí mỗi phiên), và "vào lệnh phiên
`t+1`" thực chất là mua ở giá đóng cửa `t`. `34` đã sửa cả bốn và đo từng lỗi ở
mục 10 của nó, nên ở đây ta đối chiếu thẳng với bản đã sửa.

In [6]:
def chay_backtest(
    m_gia: pd.DataFrame,
    m_mo: pd.DataFrame,
    m_gtgd: pd.DataFrame,
    *,
    so_ma: int = SO_MA_GIU,
    cua_so: int = CUA_SO_DA,
    nguong_gtgd: float = NGUONG_GTGD,
    chi_phi: float = CHI_PHI_MOT_CHIEU,
    loc_nhin_truoc: bool = False,
    khop: str = "mo_cua_sau",
    lap_phien_ngung: bool = True,
    trong_so_troi: bool = True,
) -> dict:
    """Backtest đà tăng, tái cơ cấu hàng tháng, phân bổ đều lúc vào lệnh.

    Ba công tắc cho ba lỗi cần đo:

    ``loc_nhin_truoc=True`` dùng GTGD bình quân **cả kỳ** để lọc vũ trụ — tức là
    dùng thông tin của tương lai.

    ``khop`` là giá khớp. ``"mo_cua_sau"``: giá mở cửa phiên t+1 — sớm nhất có
    thể khi tín hiệu chỉ có sau giờ đóng cửa phiên t. ``"dong_cua_tin_hieu"``:
    giá đóng cửa của chính phiên t — mức giá mà tín hiệu dùng để tính.

    ``chi_phi=0`` bỏ qua chi phí.

    Hai công tắc còn lại chỉ để đo lỗi của phiên bản trước (mục 10):
    ``lap_phien_ngung=False`` không lấp giá qua phiên tạm ngừng,
    ``trong_so_troi=False`` giữ trọng số cố định 1/N mỗi phiên.
    """
    ngay_tai_co_cau = m_gia.resample(TAN_SUAT_TAI_CO_CAU).last().index
    ngay_tai_co_cau = [d for d in ngay_tai_co_cau if (m_gia.index <= d).any()]

    # Phiên tạm ngừng: giá đứng yên ở giá cuối cùng, để phiên giao dịch lại mang
    # trọn biến động của cả khoảng ngừng. pandas 3 bỏ fill_method của pct_change(),
    # nên không ffill thì ô trống ra NaN và phiên mở lại cũng ra NaN.
    gia_dong = m_gia.ffill() if lap_phien_ngung else m_gia
    gia_mo = m_mo.fillna(gia_dong.shift(1))  # không có giá mở cửa → coi như mở bằng giá hôm trước
    ls_ngay = gia_dong.pct_change().fillna(0).to_numpy()
    qua_dem = (gia_mo / gia_dong.shift(1) - 1).fillna(0).to_numpy()  # đóng cửa t → mở cửa t+1
    trong_phien = (gia_dong / gia_mo - 1).fillna(0).to_numpy()  # mở cửa → đóng cửa
    gtgd_ca_ky = m_gtgd.mean()

    danh_muc_theo_ky: dict[pd.Timestamp, list[str]] = {}
    lenh: dict[int, np.ndarray] = {}  # vị trí phiên đổi danh mục → trọng số mục tiêu
    truoc_do: set[str] = set()
    quay_vong = []

    for moc in ngay_tai_co_cau:
        # Phiên giao dịch cuối cùng KHÔNG SAU mốc — đây là phiên ta biết mọi thứ
        co = m_gia.index[m_gia.index <= moc]
        if len(co) < cua_so + 2:
            continue
        t = co[-1]

        # ── Bộ lọc vũ trụ ────────────────────────────────────────────────
        if loc_nhin_truoc:
            du_thanh_khoan = gtgd_ca_ky[gtgd_ca_ky >= nguong_gtgd].index  # ⚠️ dùng tương lai
        else:
            cua_so_gtgd = m_gtgd.loc[:t].tail(cua_so)
            bq = cua_so_gtgd.mean()
            du_thanh_khoan = bq[bq >= nguong_gtgd].index

        # ── Tín hiệu: lợi suất `cua_so` phiên tính tới t ──────────────────
        lich_su = m_gia.loc[:t]
        da_tang = lich_su.iloc[-1] / lich_su.iloc[-1 - cua_so] - 1
        du_dieu_kien = da_tang[da_tang.index.isin(du_thanh_khoan)].dropna()
        chon = du_dieu_kien.nlargest(so_ma).index.tolist()
        if not chon:
            continue
        danh_muc_theo_ky[t] = chon

        # Danh mục mới bắt đầu sinh lời từ phiên t+1 — mua ở giá đóng cửa t,
        # hoặc ở giá mở cửa t+1, tuỳ `khop`.
        vi_tri = m_gia.index.get_loc(t) + 1
        if vi_tri >= len(m_gia.index):
            continue
        w = np.zeros(m_gia.shape[1])
        w[m_gia.columns.get_indexer(chon)] = 1 / len(chon)
        lenh[vi_tri] = w

        moi = set(chon)
        quay_vong.append(len(moi ^ truoc_do) / (2 * len(moi)) if truoc_do else 1.0)
        truoc_do = moi

    # ── Lợi suất danh mục, từng phiên ────────────────────────────────────
    w = np.zeros(m_gia.shape[1])  # trọng số lúc đóng cửa phiên trước; tổng 0 = toàn tiền mặt
    ls_dm = np.full(len(m_gia.index), np.nan)
    for j in range(min(lenh, default=len(ls_dm)), len(ls_dm)):
        if j in lenh:
            moi = lenh[j]
            if khop == "dong_cua_tin_hieu":
                doi = np.abs(moi - w).sum()  # đổi ở giá đóng cửa phiên trước
                ls_dm[j] = (1 - doi * chi_phi) * (1 + moi @ ls_ngay[j]) - 1
                w, r = moi, ls_ngay[j]
            else:
                # Qua đêm vẫn là danh mục cũ; đổi danh mục ở giá mở cửa.
                gia_tri_mo = 1 - w.sum() + w @ (1 + qua_dem[j])
                w_mo = w * (1 + qua_dem[j]) / gia_tri_mo if trong_so_troi else w
                doi = np.abs(moi - w_mo).sum()
                ls_dm[j] = gia_tri_mo * (1 - doi * chi_phi) * (1 + moi @ trong_phien[j]) - 1
                w, r = moi, trong_phien[j]
        else:
            ls_dm[j] = w @ ls_ngay[j]
            r = ls_ngay[j]
        if trong_so_troi and w.sum() > 0:
            w = w * (1 + r) / (w @ (1 + r))  # trọng số trôi theo giá

    return {
        "loi_suat": pd.Series(ls_dm, index=m_gia.index).dropna(),
        "danh_muc": danh_muc_theo_ky,
        "quay_vong": float(np.mean(quay_vong)) if quay_vong else np.nan,
    }

In [7]:
def lich_tai_co_cau(lich: pd.DatetimeIndex) -> list[pd.Timestamp]:
    # Cùng lịch với chay_backtest: phiên cuối cùng không sau cuối mỗi tháng. Lịch
    # nghỉ của Sở công bố trước, nên biết "hôm nay là phiên cuối tháng" không phải nhìn trước.
    moc = pd.Series(lich, index=lich).resample(TAN_SUAT_TAI_CO_CAU).last().index
    return [lich[lich <= m][-1] for m in moc if (lich <= m).any()]


def do_luong(ls: pd.Series, ten: str = "") -> dict:
    # Cùng công thức với 34: CAGR theo năm lịch, Sharpe = CAGR / biến động, rf = 0.
    ls = ls.dropna()
    von = (1 + ls).cumprod()
    so_nam = (ls.index[-1] - ls.index[0]).days / 365.25
    cagr = von.iloc[-1] ** (1 / so_nam) - 1
    bien_dong = ls.std() * np.sqrt(252)
    return {
        "chiến lược": ten,
        "CAGR %": round(cagr * 100, 2),
        "Biến động %": round(bien_dong * 100, 2),
        "Sharpe": round(cagr / bien_dong, 3) if bien_dong > 0 else np.nan,
        "Sụt sâu nhất %": round((von / von.cummax() - 1).min() * 100, 2),
        "Số phiên": len(ls),
    }


BAN_34 = chay_backtest(M_GIA, M_MO, M_GTGD)
RO_34 = BAN_34["danh_muc"]
LS_34 = BAN_34["loi_suat"]
BAT_DAU = LS_34.index[0]
print(f"{len(RO_34)} kỳ tái cơ cấu · nắm giữ từ {BAT_DAU:%d/%m/%Y} · quay vòng bình quân {BAN_34['quay_vong']:.0%}")
pd.DataFrame([do_luong(LS_34, "34 — vectorised, bản đúng")])

58 kỳ tái cơ cấu · nắm giữ từ 07/02/2022 · quay vòng bình quân 56%


,chiến lược,CAGR %,Biến động %,Sharpe,Sụt sâu nhất %,Số phiên
0,"34 — vectorised, bản đúng",-3.93,27.99,-0.141,-55.05,1173


Con số này là **thước chuẩn** cho cả notebook: cùng dữ liệu, cùng ngày chạy thì
nó phải trùng với bảng "bản đúng" của `34`.

## 3 · Nhiều feed trong một `Cerebro`

Mỗi mã là một feed. backtrader đồng bộ các feed theo thời gian: ở mỗi bước, feed
nào có thanh mang mốc thời gian đó thì tiến lên một thanh, feed nào không có thì
**đứng yên**. Ba hệ quả mà chiến lược nhiều mã phải biết:

| Hiện tượng | Biểu hiện trong strategy |
|---|---|
| Mã **niêm yết muộn** | chưa có thanh nào → `len(data) == 0`; chừng nào còn một feed chưa đủ `minperiod`, backtrader gọi `prenext()` chứ **không** gọi `next()` |
| Mã **tạm ngừng** giao dịch | `len(data)` không tăng; `data.close[0]` là giá **phiên cũ**; `data.datetime.date(0)` khác ngày hiện tại |
| Lần đầu mọi feed đủ dữ liệu | `nextstart()` được gọi đúng một lần, rồi mới tới `next()` |

Xem tận mắt trên ba mã: một mã có đủ lịch sử, mã niêm yết muộn nhất, và mã có
khoảng tạm ngừng dài nhất trong vũ trụ.

In [8]:
co_gia = M_GIA.notna()
ngay_dau = co_gia.idxmax()
khoang_trong = {
    m: int((~co_gia.loc[ngay_dau[m] :, m]).sum())
    for m in M_GIA.columns
    if co_gia[m].sum() > 250 and co_gia[m].iloc[-1]
}
MA_DU = M_GTGD.mean().idxmax()  # thanh khoản nhất, có đủ lịch sử
MA_MUON = ngay_dau[ngay_dau < M_GIA.index[-250]].idxmax()  # niêm yết muộn nhất (còn ≥ 1 năm dữ liệu)
MA_NGUNG = max(khoang_trong, key=khoang_trong.get)  # nhiều phiên trống nhất sau khi niêm yết
print(f"Đủ lịch sử: {MA_DU} · niêm yết muộn: {MA_MUON} (từ {ngay_dau[MA_MUON]:%d/%m/%Y}) · "
      f"tạm ngừng: {MA_NGUNG} ({khoang_trong[MA_NGUNG]} phiên trống)")


class NhatKy(bt.Strategy):
    # Ghi lại backtrader gọi hàm nào ở mỗi bước và mỗi feed đang đứng ở đâu.
    def __init__(self):
        self.dong = []

    def _ghi(self, giai_doan):
        hom_nay = self.datetime.date(0)
        self.dong.append(
            {"date": pd.Timestamp(hom_nay), "giai_doan": giai_doan}
            | {f"len({d._name})": len(d) for d in self.datas}
            | {f"{d._name} có thanh hôm nay": len(d) > 0 and d.datetime.date(0) == hom_nay for d in self.datas}
        )

    def prenext(self):
        self._ghi("prenext")

    def nextstart(self):
        self._ghi("nextstart")
        self.next()  # mặc định nextstart() cũng chỉ gọi next(); ghi đè để thấy nó

    def next(self):
        if not self.dong or self.dong[-1]["date"] != pd.Timestamp(self.datetime.date(0)):
            self._ghi("next")


KHUNG = {m: k for m, k in gia.groupby("symbol", observed=True)}  # tách một lần, dùng lại cho mọi lần chạy
cerebro = bt.Cerebro(stdstats=False)
for m in [MA_DU, MA_MUON, MA_NGUNG]:
    cerebro.adddata(feed_tu_finlens(KHUNG[m], don_vi_gia=DON_VI_GIA))
cerebro.addstrategy(NhatKy)
nhat_ky = pd.DataFrame(cerebro.run()[0].dong)

print(nhat_ky["giai_doan"].value_counts().to_string())
nhat_ky[nhat_ky["giai_doan"] == "nextstart"]

Đủ lịch sử: HPG · niêm yết muộn: CRV (từ 13/10/2025) · tạm ngừng: GEE (18 phiên trống)


giai_doan
prenext      992
next         252
nextstart      1


,date,giai_doan,len(HPG),len(CRV),len(GEE),HPG có thanh hôm nay,CRV có thanh hôm nay,GEE có thanh hôm nay
992,2025-10-13,nextstart,993,1,882,True,True,True


Chừng nào mã niêm yết muộn chưa có thanh đầu tiên, **mọi** bước đều là
`prenext` — kể cả khi hai mã kia đã giao dịch hàng năm trời. Một strategy chỉ
viết `next()` sẽ ngồi im suốt khoảng đó mà không báo gì. Mục 6 đo cái giá của
việc này trên cả vũ trụ.

Còn đây là quanh khoảng tạm ngừng dài nhất của mã thứ ba: `len()` đứng yên,
còn feed thì vẫn trả giá của phiên cũ.

In [9]:
cot_len = f"len({MA_NGUNG})"
cot_co = f"{MA_NGUNG} có thanh hôm nay"
dung_yen = nhat_ky[~nhat_ky[cot_co] & (nhat_ky[cot_len] > 0)]
if dung_yen.empty:
    print("Không có phiên trống nào sau ngày niêm yết")
else:
    # Khoảng trống dài nhất: nhóm các phiên trống liền nhau
    nhom = (nhat_ky[cot_co] != nhat_ky[cot_co].shift()).cumsum()
    dai_nhat = dung_yen.groupby(nhom[dung_yen.index]).size().idxmax()
    vi_tri = nhat_ky.index[nhom == dai_nhat]
    cua_so = nhat_ky.loc[vi_tri[0] - 2 : vi_tri[-1] + 2, ["date", cot_len, cot_co]]
    print(f"{len(vi_tri)} phiên liền nhau {MA_NGUNG} không có thanh; len() đứng ở {nhat_ky.loc[vi_tri[0], cot_len]}")
    display(pd.concat([cua_so.head(5), cua_so.tail(4)]))

18 phiên liền nhau GEE không có thanh; len() đứng ở 592


,date,len(GEE),GEE có thanh hôm nay
683,2024-07-17,591,True
684,2024-07-18,592,True
685,2024-07-19,592,False
686,2024-07-22,592,False
687,2024-07-23,592,False
701,2024-08-12,592,False
702,2024-08-13,592,False
703,2024-08-14,593,True
704,2024-08-15,594,True


Vì thế chiến lược nhiều mã phải tự hỏi **từng feed** hai câu trước khi dùng nó:
`len(d) > 0` (đã niêm yết chưa) và `d.datetime.date(0) == self.datetime.date(0)`
(hôm nay có giao dịch không). Và phải cho `prenext()` gọi `next()`.

### Giới hạn vũ trụ: chỉ nạp mã có thể được chọn

`34` quét cả sàn, nhưng mã nào **chưa từng** qua bộ lọc thanh khoản ở bất kỳ
ngày tái cơ cấu nào thì không bao giờ vào rổ — nạp nó vào `Cerebro` chỉ tốn
thời gian. Bộ lọc dưới đây không nhìn trước: nó chỉ loại những feed mà chiến
lược, chạy đúng luật, chắc chắn không chạm tới. Mục 7 chứng minh kết quả không
đổi và đo thời gian tiết kiệm được.

Có hai cách đo "GTGD bình quân 60 phiên": 60 phiên **của thị trường** (cách của
`34`, ô trống bị bỏ qua) và 60 thanh **của chính mã** (cách tự nhiên trong
backtrader). Vũ trụ lấy hợp của cả hai.

In [10]:
KY_TCC = lich_tai_co_cau(M_GIA.index)
gtgd_60_thi_truong = M_GTGD.rolling(CUA_SO_DA, min_periods=1).mean()
gtgd_60_cua_ma = gia.groupby("symbol", observed=True)["gtgd"].transform(lambda s: s.rolling(CUA_SO_DA, min_periods=1).mean())
gtgd_60_cua_ma = gia.assign(g=gtgd_60_cua_ma).pivot(index="date", columns="symbol", values="g")

du_o_ky = (gtgd_60_thi_truong.loc[KY_TCC] >= NGUONG_GTGD) | (gtgd_60_cua_ma.reindex(KY_TCC) >= NGUONG_GTGD)
VU_TRU = sorted(du_o_ky.columns[du_o_ky.any()])
TAT_CA = sorted(KHUNG)
print(f"Vũ trụ nạp vào Cerebro: {len(VU_TRU)} / {len(TAT_CA)} mã từng qua bộ lọc thanh khoản ở một kỳ tái cơ cấu")

Vũ trụ nạp vào Cerebro: 294 / 409 mã từng qua bộ lọc thanh khoản ở một kỳ tái cơ cấu


## 4 · Chiến lược đà tăng trong backtrader

Một strategy, nhiều công tắc — mỗi công tắc là một giả định mà ma trận của
`34` ngầm chọn sẵn. Mục 6 bật/tắt từng cái để đo.

| Tham số | Ý nghĩa |
|---|---|
| `ro` | rổ đã chọn sẵn (`dict` ngày → danh sách mã, như `danh_muc` của `chay_backtest`), hoặc `None` để strategy **tự tính** tín hiệu trên dữ liệu feed |
| `ty_le` | phần giá trị danh mục được đầu tư — phần còn lại là tiền mặt đệm cho phí và khoảng nhảy giá |
| `lo` | `1` dùng `order_target_percent` (cổ phiếu nguyên); `100` làm tròn xuống lô bằng `order_target_size` |
| `ban_truoc` | đặt lệnh bán trước lệnh mua |
| `goi_next_o_prenext` | cho `prenext()` gọi `next()` |

**Vì sao bán trước mua sau.** Broker kiểm tiền ngay lúc nhận lệnh (`checksubmit`)
bằng cách giả-khớp các lệnh **theo thứ tự đặt**. Lệnh bán đặt trước thì tiền
bán được tính vào sức mua của các lệnh mua đặt sau; đặt ngược lại thì lệnh mua
bị từ chối `Margin` vì tiền vẫn đang nằm trong cổ phiếu cũ. Đến lúc khớp thật,
broker cũng xử lý theo thứ tự đó.

In [11]:
class DaTang(bt.Strategy):
    params = dict(
        ngay_tcc=(),
        ro=None,
        ty_le=0.995,
        lo=1,
        ban_truoc=True,
        goi_next_o_prenext=True,
    )

    def __init__(self):
        self.tcc = {pd.Timestamp(t).date() for t in self.p.ngay_tcc}
        self.so_buoc = 0
        self.so_prenext = 0
        self.so_next = 0
        self.tu_choi = 0
        self.ro_da_chon = {}
        self.ty_le_tien = []

    def notify_order(self, order):
        if order.status in (order.Margin, order.Rejected):
            self.tu_choi += 1

    def prenext(self):
        self.so_prenext += 1
        if self.p.goi_next_o_prenext:
            self._tu_prenext = True
            self.next()
            self._tu_prenext = False

    def next(self):
        self.so_buoc += 1
        self.so_next += not getattr(self, "_tu_prenext", False)  # chỉ đếm lần backtrader tự gọi next()
        self.ty_le_tien.append(self.broker.getcash() / self.broker.getvalue())
        hom_nay = self.datetime.date(0)
        if hom_nay not in self.tcc or self.so_buoc < CUA_SO_DA + 2:
            return

        chon = self.p.ro.get(pd.Timestamp(hom_nay), []) if self.p.ro is not None else self.tu_chon(hom_nay)
        if not chon:
            return
        self.ro_da_chon[pd.Timestamp(hom_nay)] = chon

        gia_tri = self.broker.getvalue()
        ban, mua = [], []
        for d in self.datas:
            phan_tram = self.p.ty_le / len(chon) if d._name in chon else 0.0
            hien_co = self.getposition(d).size
            if self.p.lo == 1:
                lenh = (self.order_target_percent, d, phan_tram)
                muc_tieu = phan_tram * gia_tri / d.close[0] if len(d) else 0
            else:
                muc_tieu = lam_tron_lo(phan_tram * gia_tri / d.close[0], self.p.lo) if len(d) else 0
                lenh = (self.order_target_size, d, muc_tieu)
            if int(muc_tieu) < hien_co:
                ban.append(lenh)
            elif int(muc_tieu) > hien_co:
                mua.append(lenh)
        for ham, d, muc_tieu in (ban + mua if self.p.ban_truoc else mua + ban):
            ham(data=d, target=muc_tieu)

    def tu_chon(self, hom_nay):
        # Tín hiệu tính trên chính feed: 60 thanh GẦN NHẤT CỦA MÃ, không phải 60 phiên của thị trường.
        diem = {}
        for d in self.datas:
            if len(d) <= CUA_SO_DA or d.datetime.date(0) != hom_nay:
                continue  # chưa đủ lịch sử, hoặc hôm nay mã không giao dịch
            gtgd = np.mean([d.close[-k] * d.volume[-k] for k in range(CUA_SO_DA)])
            if gtgd >= NGUONG_GTGD:
                diem[d._name] = d.close[0] / d.close[-CUA_SO_DA] - 1
        return sorted(diem, key=diem.get, reverse=True)[:SO_MA_GIU]

Hàm chạy dùng chung. Hai núm nằm ở broker chứ không ở strategy:

- Mặc định của backtrader là khớp lệnh Market ở **giá mở cửa phiên sau** — thứ
  duy nhất làm được ngoài đời khi tín hiệu chỉ có sau giờ đóng cửa, và cũng là
  bản đúng của `34`. `khop_dong_cua=True` bật `set_coc` — khớp ở **giá đóng cửa
  phiên đặt lệnh**, tương đương `khop="dong_cua_tin_hieu"` của `34`. Mục 6 dùng
  nó để kiểm hai bộ máy.
- `phi`, `thue_ban` đi vào `PhiCoPhieuVN`. Đặt `phi=0.0018, thue_ban=0` là chép
  giả định 0,18% mỗi chiều của `34`.

In [12]:
def chay(ma_ds=VU_TRU, *, khop_dong_cua=False, phi=0.0015, thue_ban=0.001, von=VON, **tham_so) -> dict:
    cerebro = bt.Cerebro(stdstats=False)
    for m in ma_ds:
        cerebro.adddata(feed_tu_finlens(KHUNG[m], don_vi_gia=DON_VI_GIA))
    cerebro.broker.setcash(von)
    cerebro.broker.set_coc(khop_dong_cua)
    cerebro.broker.addcommissioninfo(PhiCoPhieuVN(phi=phi, thue_ban=thue_ban))
    them_analyzer(cerebro)
    cerebro.addstrategy(DaTang, ngay_tcc=KY_TCC, **tham_so)
    t0 = time.perf_counter()
    strat = cerebro.run()[0]
    kq = ket_qua(strat)
    return {
        "strat": strat,
        "kq": kq,
        "giay": time.perf_counter() - t0,
        "ls": kq["loi_suat"].loc[BAT_DAU:],  # cùng khoảng với thước chuẩn
        "gia_tri_cuoi": strat.broker.getvalue(),
    }

## 5 · Đối chiếu hai đường vốn

Bản backtrader "theo luật Việt Nam": tự tính tín hiệu, khớp ở giá mở cửa phiên
sau, lô 100, phí 0,15% hai chiều + thuế bán 0,1%, giữ 0,5% tiền mặt đệm.

In [13]:
VN = chay(lo=100)
print(f"{len(VU_TRU)} feed · {VN['giay']:.0f} giây · {VN['strat'].tu_choi} lệnh bị từ chối · "
      f"{VN['kq']['tom_tat']['so_lenh_khop']:,} lần khớp")
pd.DataFrame([do_luong(LS_34, "34 — vectorised"), do_luong(VN["ls"], "backtrader — luật VN")])

294 feed · 65 giây · 0 lệnh bị từ chối · 1,229 lần khớp


,chiến lược,CAGR %,Biến động %,Sharpe,Sụt sâu nhất %,Số phiên
0,34 — vectorised,-3.93,27.99,-0.141,-55.05,1173
1,backtrader — luật VN,-4.65,27.25,-0.171,-54.22,1173


In [14]:
def duong_von(ls: pd.Series, ten: str) -> pd.DataFrame:
    von = (1 + ls).cumprod() * 100
    return pd.DataFrame({"date": von.index, "von": von.to_numpy(), "duong": ten})


duong(
    pd.concat([duong_von(LS_34, "34 — vectorised"), duong_von(VN["ls"], "backtrader — luật VN")]),
    x="date",
    y="von",
    theo="duong",
    tieu_de="Cùng chiến lược, hai bộ máy",
    phu_de="Gốc 100 tại phiên nắm giữ đầu tiên · tái cơ cấu cuối mỗi tháng",
    nhan_y="giá trị danh mục (gốc = 100)",
)

Khoảng cách CAGR là **−0,72 điểm %** (−3,93% so với −4,65%). Hai đường vốn cùng
hình dạng — sụt sâu nhất gần như bằng nhau (−55,05% và −54,22%) — nhưng đường
backtrader nằm thấp hơn đều. Mục 6 tách khoảng cách đó ra từng nguồn.

## 6 · Truy từng nguồn lệch

**Kiểm bộ máy trước.** Nếu hai bộ máy cùng giả định mà ra kết quả khác nhau thì
mọi phân tích phía sau vô nghĩa. Giả định dễ dựng giống hệt nhất ở cả hai bên là
khớp ở giá đóng cửa phiên tín hiệu: `khop="dong_cua_tin_hieu"` bên `34`,
`set_coc(True)` bên backtrader — cùng rổ, cùng 0,18% mỗi chiều.

In [15]:
GIONG_34 = dict(ro=RO_34, phi=CHI_PHI_MOT_CHIEU, thue_ban=0.0)
dong_cua_34 = chay_backtest(M_GIA, M_MO, M_GTGD, khop="dong_cua_tin_hieu")["loi_suat"]
dong_cua_bt = chay(khop_dong_cua=True, **GIONG_34)

KIEM_MAY = pd.DataFrame([
    do_luong(dong_cua_34, "34 — khớp đóng cửa t"),
    do_luong(dong_cua_bt["ls"], "backtrader — set_coc(True), cùng rổ"),
]).set_index("chiến lược")
print(f"Lệch CAGR giữa hai bộ máy cùng giả định: {KIEM_MAY['CAGR %'].diff().iloc[-1]:+.2f} điểm % · "
      f"{dong_cua_bt['strat'].tu_choi} lệnh bị từ chối · tiền mặt bq "
      f"{np.mean(dong_cua_bt['strat'].ty_le_tien[-len(dong_cua_bt['ls']):]):.2%}")
KIEM_MAY[["CAGR %", "Sharpe", "Sụt sâu nhất %"]]

Lệch CAGR giữa hai bộ máy cùng giả định: +0.02 điểm % · 0 lệnh bị từ chối · tiền mặt bq 0.31%


,CAGR %,Sharpe,Sụt sâu nhất %
chiến lược,,,
34 — khớp đóng cửa t,-3.28,-0.117,-55.94
"backtrader — set_coc(True), cùng rổ",-3.26,-0.117,-55.83


**Cùng giả định, hai bộ máy lệch +0,02 điểm %** — nhỏ hơn mọi mức nhiễu. Không
lệnh nào bị từ chối, vì lệnh khớp ở đúng mức giá dùng để tính khối lượng; tiền
mặt bình quân 0,31% là phần còn lại của khoản đệm 0,5%. Vậy phần lệch phía sau
**không đến từ bộ máy mà đến từ giả định**.

Giờ đi từ bản đúng của `34` tới bản luật VN **từng bước một**, mỗi bước đổi
đúng một giả định. Hiệu CAGR giữa hai bước liền nhau là đóng góp của giả định
đó; cộng dồn lại đúng bằng khoảng cách tổng.

| Bước | Đổi gì so với bước trước |
|---|---|
| 0 | `34` bản đúng (vectorised) |
| 1 | sang **backtrader**, giữ mọi giả định của `34`: rổ của `34`, khớp giá mở cửa phiên sau, 0,18% mỗi chiều, cổ phiếu nguyên. Khác biệt duy nhất là thứ ma trận không có: lệnh có **khối lượng** tính từ giá đóng cửa `t` nhưng khớp ở giá mở cửa `t+1`, nên phải chừa **0,5% tiền mặt** và vẫn có thể **bị từ chối vì thiếu tiền** |
| 2 | strategy **tự tính tín hiệu** trên feed |
| 3 | làm tròn **lô 100** |
| 4 | **phí 0,15% hai chiều + thuế bán 0,1%** thay cho 0,18% mỗi chiều |

In [16]:
GIONG_34_TU_TINH = dict(phi=CHI_PHI_MOT_CHIEU, thue_ban=0.0)
SANG_BT = chay(**GIONG_34)
TU_TINH = chay(**GIONG_34_TU_TINH)
LO_100 = chay(lo=100, **GIONG_34_TU_TINH)

buoc = [
    ("0 · 34 bản đúng", LS_34),
    ("1 · sang backtrader (tiền mặt dư 0,5%)", SANG_BT),
    ("2 · tự tính tín hiệu trên feed", TU_TINH),
    ("3 · lô 100", LO_100),
    ("4 · phí 0,15% + thuế bán 0,1%", VN),
]
dong = []
for ten, x in buoc:
    ls = x if isinstance(x, pd.Series) else x["ls"]
    d = do_luong(ls, ten)
    d["lệnh bị từ chối"] = x["strat"].tu_choi if isinstance(x, dict) else 0
    d["tiền mặt bq %"] = round(np.mean(x["strat"].ty_le_tien[-len(ls):]) * 100, 2) if isinstance(x, dict) else 0.0
    dong.append(d)
CHUOI_LECH = pd.DataFrame(dong).set_index("chiến lược")
CHUOI_LECH["Δ CAGR (điểm %)"] = CHUOI_LECH["CAGR %"].diff().round(2)
CHUOI_LECH[["CAGR %", "Δ CAGR (điểm %)", "Sharpe", "Sụt sâu nhất %", "lệnh bị từ chối", "tiền mặt bq %"]]

,CAGR %,Δ CAGR (điểm %),Sharpe,Sụt sâu nhất %,lệnh bị từ chối,tiền mặt bq %
chiến lược,,,,,,
0 · 34 bản đúng,-3.93,NaN,-0.141,-55.05,0,0.00
"1 · sang backtrader (tiền mặt dư 0,5%)",-4.08,-0.15,-0.149,-54.40,16,1.92
2 · tự tính tín hiệu trên feed,-4.57,-0.49,-0.166,-54.40,17,1.85
3 · lô 100,-4.25,0.32,-0.156,-54.10,0,3.32
"4 · phí 0,15% + thuế bán 0,1%",-4.65,-0.40,-0.171,-54.22,0,3.28


Trước khi đọc bảng, cần biết **mức nhiễu nền**: một danh mục 15 mã làm tròn cổ
phiếu đổi kết quả chỉ vì vốn ban đầu khác đi một chút — số cổ phiếu lẻ, lệnh nào
vừa đủ tiền, lệnh nào bị từ chối đều đổi theo. Chạy lại bản luật VN với vốn
lệch 1% và 2%:

In [17]:
nhieu = pd.DataFrame(
    [do_luong(VN["ls"], f"vốn {VON / 1e9:.2f} tỷ")]
    + [do_luong(chay(lo=100, von=VON * k)["ls"], f"vốn {VON * k / 1e9:.2f} tỷ") for k in (1.01, 1.02)]
)
NHIEU_NEN = nhieu["CAGR %"].max() - nhieu["CAGR %"].min()
print(f"Khoảng dao động CAGR chỉ vì vốn lệch ≤ 2%: {NHIEU_NEN:.2f} điểm %")
nhieu[["chiến lược", "CAGR %", "Sharpe"]]

Khoảng dao động CAGR chỉ vì vốn lệch ≤ 2%: 0.24 điểm %


,chiến lược,CAGR %,Sharpe
0,vốn 1.00 tỷ,-4.65,-0.171
1,vốn 1.01 tỷ,-4.41,-0.162
2,vốn 1.02 tỷ,-4.62,-0.170


In [18]:
dong_gop = CHUOI_LECH["Δ CAGR (điểm %)"].dropna().rename_axis("nguồn").reset_index()
dong_gop["nguồn"] = dong_gop["nguồn"].str.split(" · ").str[1]
tong = CHUOI_LECH["CAGR %"].iloc[-1] - CHUOI_LECH["CAGR %"].iloc[0]
dong_gop = pd.concat(
    [dong_gop, pd.DataFrame([{"nguồn": "TỔNG — 34 → luật VN", "Δ CAGR (điểm %)": round(tong, 2)}])],
    ignore_index=True,
)
dong_gop["vượt nhiễu nền?"] = np.where(dong_gop["Δ CAGR (điểm %)"].abs() > NHIEU_NEN, "có", "không")
print(f"Tổng đóng góp các bước = {dong_gop['Δ CAGR (điểm %)'].iloc[:-1].sum():+.2f} điểm % "
      f"(khoảng cách đo trực tiếp {tong:+.2f})")
dong_gop

Tổng đóng góp các bước = -0.72 điểm % (khoảng cách đo trực tiếp -0.72)


,nguồn,Δ CAGR (điểm %),vượt nhiễu nền?
0,"sang backtrader (tiền mặt dư 0,5%)",-0.15,không
1,tự tính tín hiệu trên feed,-0.49,có
2,lô 100,0.32,có
3,"phí 0,15% + thuế bán 0,1%",-0.40,có
4,TỔNG — 34 → luật VN,-0.72,có


In [19]:
bar_ngang(
    dong_gop.iloc[:-1],
    nhan="nguồn",
    gia_tri="Δ CAGR (điểm %)",
    tieu_de="Nguồn lệch → đóng góp vào CAGR",
    phu_de=f"Đi từ 34 tới backtrader luật VN, mỗi bước đổi một giả định · nhiễu nền ±{NHIEU_NEN:.2f} điểm %",
    nhan_x="điểm phần trăm CAGR",
    dinh_dang_nhan="{:+.2f}",
)

**Khoảng cách tổng −0,72 điểm % lớn gấp ba nhiễu nền (0,24); ba trong bốn
thành phần vượt nhiễu nền.** Đọc từng dòng:

| Nguồn | Đóng góp | Vì sao |
|---|---|---|
| Sang backtrader | −0,15 | Ma trận của `34` đổi danh mục ở giá mở cửa với **đúng** trọng số mục tiêu. Broker thì nhận lệnh có khối lượng tính từ giá đóng cửa `t`: phiên sau mở cửa cao hơn là lệnh mua đặt sau cùng bị từ chối — 16 lệnh, mỗi lệnh để một suất 1/15 nằm im cả tháng, tiền mặt bình quân 1,92% thay vì 0 (6b). Dưới nhiễu nền |
| Tự tính tín hiệu trên feed | **−0,49** | Chỉ 3/58 kỳ chọn khác `34`, mỗi kỳ đúng một mã (6a) — nhưng rổ đà tăng toàn mã biến động mạnh, nên một mã khác trong một tháng cũng đủ |
| Phí 0,15% + thuế bán 0,1% | **−0,40** | Chiều mua rẻ hơn (0,15% so với 0,18%) nhưng chiều bán đắt hơn (0,25% so với 0,18%): mỗi vòng mua-bán tốn 0,40% thay vì 0,36%, nhân với số vòng quay của một rổ đổi mã hàng tháng. Gộp phí và thuế thành "0,18% mỗi chiều" như `34` là đánh giá thấp chi phí |
| Lô 100 | **+0,32** | Làm tròn xuống lô để lại thêm ~1,5% tiền mặt (1,85% → 3,32%), và chính khoản đệm đó xoá hết 17 lệnh bị từ chối. Với một chiến lược lỗ, tiền nằm im là lỗ ít hơn. Chỉ vừa vượt nhiễu nền |

Lô 100 kéo ngược chiều ba nguồn còn lại, nên khoảng cách tổng nhỏ hơn tổng độ
lớn các thành phần. Khoảng cách tổng nhỏ chưa bao giờ là bằng chứng hai bên
khớp nhau — mục 10 của `34` cho thấy hai lỗi lớn nhất của bản trước (trọng số
cố định và khớp giá đóng cửa) cũng ngược chiều nhau.

### 6a · Bước 2 — vì sao tự tính tín hiệu lại chọn khác

Strategy đo 60 phiên **trên thanh của chính mã**; `34` đo 60 phiên **trên lịch
thị trường**. Hai cách trùng nhau với mọi mã giao dịch đều đặn, và tách ra đúng
ở những mã có phiên trống trong cửa sổ.

In [20]:
ro_tu_tinh = TU_TINH["strat"].ro_da_chon
lech = []
for t, ro in RO_34.items():
    mot, hai = set(ro), set(ro_tu_tinh.get(t, []))
    for m in sorted(mot ^ hai):
        cua_so = M_GIA.loc[:t, m].tail(CUA_SO_DA + 1)
        lech.append({
            "kỳ": t.date(),
            "mã": m,
            "chỉ ở rổ": "34" if m in mot else "backtrader",
            "phiên có giá / 61 phiên thị trường": int(cua_so.notna().sum()),
        })
trung = np.mean([len(set(ro) & set(ro_tu_tinh.get(t, []))) / len(ro) for t, ro in RO_34.items()])
print(f"Rổ trùng nhau {trung:.2%} · {len({x['kỳ'] for x in lech})}/{len(RO_34)} kỳ có khác biệt")
pd.DataFrame(lech)

Rổ trùng nhau 99.66% · 3/58 kỳ có khác biệt


,kỳ,mã,chỉ ở rổ,phiên có giá / 61 phiên thị trường
0,2024-04-26,CTS,34,61
1,2024-04-26,VTP,backtrader,54
2,2025-07-31,VAB,backtrader,53
3,2025-07-31,VPB,34,61
4,2026-01-30,MCH,backtrader,56
5,2026-01-30,VPL,34,61


Cả ba mã chỉ backtrader chọn đều **thiếu giá** ở 5–8 trong 61 phiên thị trường
gần nhất. `34` đo đà tăng từ giá đúng 60 phiên *thị trường* trước — ô đó trống
hoặc cửa sổ thưa thì mã bị loại hay xếp khác. Strategy đo từ thanh thứ 60 *của
mã*, nằm xa hơn trên lịch, nên ra một con số đà tăng khác. Không cách nào "sai":
đó là hai định nghĩa khác nhau của "60 phiên", và chọn định nghĩa nào phải là
quyết định có chủ ý chứ không phải tác dụng phụ của bộ máy.

### 6b · Bước 1 — tiền mặt dư đổi lấy lệnh bị từ chối

Lệnh Market đặt lúc đóng cửa `t` được tính khối lượng theo giá đóng cửa `t`,
nhưng khớp ở giá mở cửa `t+1`. Phiên sau mở cửa cao hơn thì tổng tiền mua vượt
tiền có, và lệnh mua đặt sau cùng bị broker **từ chối cả lệnh** (`Margin`) —
mã đó vắng mặt khỏi rổ cả tháng. Đệm tiền mặt càng dày càng ít lệnh bị từ chối,
nhưng càng nhiều tiền nằm im.

In [21]:
dem = [(ty, chay(ty_le=ty, **GIONG_34)) for ty in (1.0, 0.98)]
dem.insert(1, (0.995, SANG_BT))
pd.DataFrame([
    do_luong(x["ls"], f"đầu tư {ty:.1%}")
    | {"lệnh bị từ chối": x["strat"].tu_choi, "tiền mặt bq %": round(np.mean(x["strat"].ty_le_tien[-len(x["ls"]):]) * 100, 2)}
    for ty, x in dem
])[["chiến lược", "CAGR %", "Sharpe", "lệnh bị từ chối", "tiền mặt bq %"]]

,chiến lược,CAGR %,Sharpe,lệnh bị từ chối,tiền mặt bq %
0,đầu tư 100.0%,-4.53,-0.173,59,6.03
1,đầu tư 99.5%,-4.08,-0.149,16,1.92
2,đầu tư 98.0%,-4.22,-0.154,1,1.84


Không mức đệm nào miễn phí. Đầu tư 100%: **59 lệnh bị từ chối**, mỗi lệnh để
một suất 1/15 nằm im cả tháng, nên tiền mặt bình quân lên tới 6,03% — nhiều hơn
mọi mức đệm chủ động. Đệm 2%: còn 1 lệnh, nhưng tiền mặt cũng ~1,8%. Đệm 0,5% ở
giữa: 16 lệnh. CAGR không đi theo thứ tự của mức đệm (−4,53 / −4,08 / −4,22) —
vì *mã nào* bị từ chối là ngẫu nhiên theo khoảng nhảy giá mở cửa, đúng loại
nhiễu mà mục trên đã đo.

Ma trận của `34` không có vấn đề này vì nó không có khối lượng: nó đổi trọng số
ngay ở giá mở cửa. Ngoài đời không ai đặt lệnh Market lúc 15:00 để khớp lúc
9:15 với khối lượng tính từ giá hôm qua mà không chừa đệm. Thông số này phải
được quyết định và ghi ra, như chi phí.

### 6c · Hai lỗi không nằm trong chuỗi: mua trước bán sau, và quên `prenext`

Hai lỗi này không phải "giả định khác" mà là **code sai** — chạy được, không
báo lỗi, ra một đường vốn trông bình thường.

In [22]:
mua_truoc = chay(lo=100, ban_truoc=False)
quen_prenext = chay(lo=100, goi_next_o_prenext=False)

loi = pd.DataFrame([
    do_luong(VN["ls"], "Đúng — bán trước, prenext gọi next"),
    do_luong(mua_truoc["ls"], "SAI — mua trước bán sau"),
    do_luong(quen_prenext["ls"], "SAI — chỉ viết next()"),
])
loi["lệnh bị từ chối"] = [x["strat"].tu_choi for x in (VN, mua_truoc, quen_prenext)]
loi["phiên strategy được gọi next"] = [x["strat"].so_next for x in (VN, mua_truoc, quen_prenext)]
muon_nhat = min(VU_TRU, key=lambda m: -KHUNG[m]["date"].min().value)
print(f"Mã niêm yết muộn nhất trong vũ trụ: {muon_nhat} (từ {KHUNG[muon_nhat]['date'].min():%d/%m/%Y})")
print(f"Chỉ viết next(): {quen_prenext['strat'].so_prenext} bước prenext bị bỏ qua, next() được gọi "
      f"{quen_prenext['strat'].so_next} lần, khớp {quen_prenext['kq']['tom_tat']['so_lenh_khop']} lệnh")
loi[["chiến lược", "CAGR %", "Sharpe", "Sụt sâu nhất %", "lệnh bị từ chối", "phiên strategy được gọi next"]]

Mã niêm yết muộn nhất trong vũ trụ: LPS (từ 18/08/2026)
Chỉ viết next(): 1203 bước prenext bị bỏ qua, next() được gọi 42 lần, khớp 0 lệnh


,chiến lược,CAGR %,Sharpe,Sụt sâu nhất %,lệnh bị từ chối,phiên strategy được gọi next
0,"Đúng — bán trước, prenext gọi next",-4.65,-0.171,-54.22,0,42
1,SAI — mua trước bán sau,-7.37,-0.332,-58.77,1336,42
2,SAI — chỉ viết next(),0.00,NaN,0.00,0,42


**Mua trước bán sau: 1.336 lệnh bị từ chối, CAGR −7,37%** thay vì −4,65%. Ở mỗi
kỳ tái cơ cấu, lệnh mua mã mới được kiểm tiền trước khi tiền bán mã cũ về, nên
hầu hết bị từ chối; danh mục chỉ mua được thứ còn tiền lẻ để mua, rồi giữ tiền
mặt cả tháng.

**Chỉ viết `next()`: 0 lệnh trong 5 năm, CAGR đúng 0.** Mã niêm yết muộn nhất
trong vũ trụ (LPS, từ 18/08/2026) giữ cả strategy ở `prenext` suốt 1.203 bước;
backtrader chỉ tự gọi `next()` 42 lần, và 42 phiên không đủ 62 phiên mà luật
cần trước kỳ tái cơ cấu đầu tiên. Cột cuối giống nhau ở cả ba dòng là vì thế —
bản đúng vẫn giao dịch suốt 5 năm vì `prenext()` gọi `next()`.

Một backtest ra 0% không phải "chiến lược hoà vốn" mà là "chiến lược chưa từng
chạy". Và lỗi này **lớn dần theo thời gian**: thêm một mã niêm yết hôm qua vào
vũ trụ là xoá sạch lịch sử của mọi mã khác.

## 7 · Thời gian chạy theo số mã

backtrader đi từng thanh của từng feed bằng Python thuần, nên thời gian tăng
theo **số feed × số phiên**. Đo trên cùng chiến lược luật VN với 20, 50, 100 mã
thanh khoản nhất, vũ trụ đã giới hạn ở mục 3, và toàn sàn.

In [23]:
theo_gtgd = M_GTGD[VU_TRU].mean().sort_values(ascending=False).index.tolist()
do_tg = []
for n in (20, 50, 100):
    x = chay(theo_gtgd[:n], lo=100)
    do_tg.append({"số feed": n, "giây": round(x["giay"], 1)})
do_tg.append({"số feed": len(VU_TRU), "giây": round(VN["giay"], 1)})
TOAN_SAN = chay(TAT_CA, lo=100)
do_tg.append({"số feed": len(TAT_CA), "giây": round(TOAN_SAN["giay"], 1)})
do_tg = pd.DataFrame(do_tg)
do_tg["ms / feed / phiên"] = (do_tg["giây"] * 1000 / do_tg["số feed"] / M_GIA.shape[0]).round(3)

print(f"Toàn sàn: giá trị cuối {TOAN_SAN['gia_tri_cuoi']:,.0f} VND · vũ trụ giới hạn: {VN['gia_tri_cuoi']:,.0f} VND "
      f"→ lệch {abs(TOAN_SAN['gia_tri_cuoi'] - VN['gia_tri_cuoi']):,.0f} VND")
do_tg

Toàn sàn: giá trị cuối 799,713,160 VND · vũ trụ giới hạn: 799,713,160 VND → lệch 0 VND


,số feed,giây,ms / feed / phiên
0,20,3.9,0.157
1,50,8.8,0.141
2,100,16.8,0.135
3,294,64.7,0.177
4,409,83.2,0.163


Thời gian **tuyến tính theo số feed**, ~0,14–0,18 ms cho mỗi feed mỗi phiên:
20 mã ~4 giây, 294 mã ~65 giây, toàn sàn 409 mã ~83 giây cho 5 năm.

Giới hạn vũ trụ ở mục 3 bỏ 115 feed, tiết kiệm khoảng một phần năm thời gian,
và **giá trị cuối khớp tới từng đồng** với lần chạy toàn sàn — bằng chứng rằng
bộ lọc đó chỉ cắt thứ chiến lược không bao giờ chạm tới. Đó là tiêu chí cho mọi
cách giới hạn vũ trụ: lọc bằng **điều kiện cần của chính luật** (ở đây là
ngưỡng thanh khoản), kiểm bằng một lần chạy đầy đủ, chứ không lọc bằng kết quả
("chỉ nạp những mã từng vào rổ" thì phải chạy xong mới biết).

Một phút cho một lần chạy là ổn để kiểm một chiến lược; là nút thắt khi quét
hàng chục bộ tham số — đúng chủ đề notebook sau.

## Tổng kết

| Bài học | Con số trong notebook này |
|---|---|
| Cùng giả định thì vectorised và hướng sự kiện cho cùng kết quả | kiểm bộ máy: +0,02 điểm % |
| Khác biệt giữa hai bộ máy là khác biệt **giả định** | `34` bản đúng → luật VN: −0,72 điểm %, gấp ba nhiễu nền |
| Lệnh có khối lượng, ma trận thì không | 16 lệnh bị từ chối ở giá mở cửa, −0,15 điểm % |
| "60 phiên" có hai định nghĩa | 3/58 kỳ chọn khác một mã, −0,49 điểm % |
| Gộp phí và thuế bán thành một tỷ lệ là đánh giá thấp chi phí | −0,40 điểm % |
| Mọi so sánh cần một mức nhiễu nền | vốn lệch ≤ 2% → CAGR dao động 0,24 điểm % |
| `attrs` mất sau `pd.concat` | `{}` — truyền `don_vi_gia=` cho `feed_tu_finlens` |
| Phiên dở dang | 406 dòng bị bỏ khi chạy lúc 13:30 |
| Bán trước, mua sau | mua trước: 1.336 lệnh bị từ chối, CAGR −7,37% |
| `prenext()` phải gọi `next()` | không gọi: 0 lệnh trong 5 năm |
| Giới hạn vũ trụ bằng điều kiện cần của luật | 294/409 feed, kết quả lệch 0 đồng |

---

Tiếp theo: [`87_toi_uu_va_walk_forward.ipynb`](87_toi_uu_va_walk_forward.ipynb) —
tối ưu tham số bằng `optstrategy`, và đo cái giá của việc chọn bộ tham số đẹp
nhất.